# PyTorch - Autograd (is what we need.)

Advanced Learning 2026





For SUBMISSION:  
  
Please upload the complete and executed `ipynb` to your git repository. Verify that all of your output can be viewed directly from github, and provide a link to that git file below.

~~~
STUDENT ID: MISSING
~~~

~~~
STUDENT GIT LINK: MISSING
~~~

In Addition, don't forget to add your ID to the files:  
`PS1_Part3_Autograd_2026_ID_[000000000].html`  



This part of the problem-set is divided into 2 sections: (a) a short introduction to PyTorch autograd, and (b) some practice questions.      
The practice questions cover calculating the gradients and updating the weights of a statistical model (linear regression), with and without PyTorch autograd.

##  A Short Intro to PyTorch Autograd

In PyTorch, autograd, short for automatic differentiation, is a core functionality that empowers you to efficiently calculate gradients during deep learning model training. It acts as a computational engine that automatically tracks the operations performed on your tensors and calculates the gradients of any loss function with respect to the model's learnable parameters.

**Why is Autograd Important?**

* Gradient Calculation: Deep learning relies heavily on gradient-based optimization algorithms like stochastic gradient descent (SGD). These algorithms adjust the weights and biases (learnable parameters) of your model based on the gradients of the loss function. Autograd automates this process, saving you from manually calculating complex derivatives.
* Efficient Backpropagation: Backpropagation is the algorithm at the heart of training deep neural networks. It propagates the error (loss) backward through the network, allowing the model to adjust its parameters in the direction that minimizes the loss. Autograd streamlines this process, making training more efficient.

**How Does Autograd Work?**

* Tracking Operations: When you enable autograd mode (with `torch.enable_grad()`), PyTorch creates a computational graph in the background. This graph tracks all the operations performed on your tensors

* Backward Pass: Once you calculate a loss function (which typically depends on the output of your model), you can call the `backward()` method on the loss tensor. This triggers the backward pass through the computational graph.

* Gradient Calculation: As the backward pass traverses the graph, autograd computes the gradients of the loss function with respect to each learnable parameter in the network. These gradients indicate how much a specific parameter contributed to the overall loss.

* Parameter Update: By utilizing these gradients, optimization algorithms like SGD update the model's learnable parameters in a direction that minimizes the loss function. This iterative process continues until the model converges (achieves a desired level of performance).

**Key Points about Autograd:**

* Automatic
* Only for Learnable Parameters: Autograd tracks gradients only for tensors that have `requires_grad=True`.
* Dynamic vs. Static Graphs (Optional): PyTorch offers both dynamic computational graphs (eager execution) and static graphs (graph mode) for model development.



In [1]:
import time
import numpy as np
import pandas as pd
import torch
import math
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using {device}!")

Using cpu!


##  0. Loading data and defining our model and loss

Load the California Housing Market dataset into a panda's DataFrame.  
In addition, scale the data (why?).

In [3]:
# Import the dataset.
cali_train_path = "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"
train_df = pd.read_csv(filepath_or_buffer=cali_train_path).drop(['longitude','latitude'],axis=1)
train_df_norm=(train_df-train_df.mean(axis=0))/train_df.std(axis=0) # column-wise operators
# Scale the labels:
print(f"Dataset dimension: {train_df_norm.shape}")
train_df_norm.head()

Dataset dimension: (17000, 7)


,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-1.079639,1.361655,1.764152,-0.361173,-0.075996,-1.252506,-1.210522
1,-0.761850,2.296540,3.230346,-0.261858,-0.099401,-1.081451,-1.096713
2,-0.920744,-0.882436,-0.866931,-0.955326,-0.999223,-1.170071,-1.048430
3,-1.159087,-0.524171,-0.480216,-0.796769,-0.715753,-0.362590,-1.154480
4,-0.682402,-0.545731,-0.506313,-0.701809,-0.622130,-1.026424,-1.222593


We define a linear model to fit the scaled median house value to our data:  

**Our model :**.    

 $$\text{median_house_value}_i=y_i = \beta_0+	\beta_1\text{housing_median_age}_i+ 	\beta_2\text{total_rooms}_i+ 	\beta_3\text{total_bedrooms}_i+ 	\beta_4\text{population}_i+ 	\beta_5\text{households}_i+ 	\beta_6\text{median_income}_i 	$$

**Our loss :**  
$$
\hat{L}_i(y_i,\hat{y}_i) = MSE(\text{y}_i,\hat{\text{y}}_i)
$$


## 1.  Manually Calculating the Gradients and Updating the Weights



Here we take a step back, and implement forward and backward propogation manually (so we can fully appreciate the autograd later...).

First, create numpy arrays for our data (X) and target (Y):

In [4]:
X = train_df_norm.drop(['median_house_value'],axis=1).to_numpy()
X =  np.c_[  np.ones(X.shape[0]), X] # adding a column of "1"s for intercept
Y = train_df_norm['median_house_value']

print(f"X dimension: {X.shape}")
print(f"Y dimension: {Y.shape}")

X dimension: (17000, 7)
Y dimension: (17000,)


### 1.1 Calculating the weights analytically (15pts)
Apply linear algebra operations to `X` and `Y` to analytically solve the linear regression problem (solve for `W`).   
Please use only numpy. Round the weights to 3 decimal places.  

In [ ]:
#### SOLUTION REQUIRED ####
t1 = time.time()

weights_rounded = # FILL IN THE MISSING CODE
print(f"Total process time: {round(time.time() - t1,3)}")
print(f"The estimated weights using the analytic solution:\n {weights_rounded.round(3)}")

Visualizing the errors:

In [ ]:
# Plot the errors and density (sanity check):
fig, (ax1,ax2) = plt.subplots(nrows=1, ncols=2,figsize=(10,4))
error = (np.matmul( X, weights_rounded)-Y)
ax1.scatter(range(Y.shape[0]), error ,s=0.05,alpha=0.7)
ax1.set_title("The error of the predictions from the truth ")
sns.kdeplot(error,ax=ax2)
ax2.set_title("The density of the error")
fig.show()


### 1.2 Calculating the weights using numpy and gradient descent (30pts)

Now you are ready to implement an algorithm that iteratively calculates the weights using numpy and gradient descent.

Please fill in your code in the `### YOUR CODE HERE` sections in the code block below.   
Please don't change the other parts of the code (unless specified).  

An epoch in machine learning means one complete pass of the training dataset through the algorithm.   

While developing your code, you may reduce the number of epochs.   
However, for final output please use the pre-defined epochs number.      





In [ ]:
#### SOLUTION REQUIRED ####

t2 = time.time()

# Randomly initialize weights using numpy
b0 = # FILL IN THE MISSING CODE
b1 = # FILL IN THE MISSING CODE
b2 = # FILL IN THE MISSING CODE
b3 = # FILL IN THE MISSING CODE
b4 = # FILL IN THE MISSING CODE
b5 = # FILL IN THE MISSING CODE
b6 = # FILL IN THE MISSING CODE

# Set learning rate (this can be a little fiddly, requires some tunning)
# Tip: start with a very small steps...

learning_rate = # FILL IN THE MISSING CODE

# Set number of epochs:
epochs = int(5e3)

# Loop through epochs, and apply gradient descent to update the weights:
for t in range(epochs):
    # Forward pass: compute predicted y
    # y_pred = b0x0 + b1x1 + b2x2 + b3x3 + b4x4 + b5x5 + b6x6
    Y_pred = # FILL IN THE MISSING CODE

    # Compute and print loss
    loss = # FILL IN THE MISSING CODE
    if t % 1000 == 0:
        print(f"Epoch:{t}, loss:{loss}")

    # Backprop to compute gradients of betas with respect to loss
    grad_y_pred = # FILL IN THE MISSING CODE
    grad_b0 = # FILL IN THE MISSING CODE
    grad_b1 = # FILL IN THE MISSING CODE
    grad_b2 = # FILL IN THE MISSING CODE
    grad_b3 = # FILL IN THE MISSING CODE
    grad_b4 = # FILL IN THE MISSING CODE
    grad_b5 = # FILL IN THE MISSING CODE
    grad_b6 = # FILL IN THE MISSING CODE

    # Update weights
    b0 -= # FILL IN THE MISSING CODE
    b1 -= # FILL IN THE MISSING CODE
    b2 -= # FILL IN THE MISSING CODE
    b3 -= # FILL IN THE MISSING CODE
    b4 -= # FILL IN THE MISSING CODE
    b5 -= # FILL IN THE MISSING CODE
    b6 -= # FILL IN THE MISSING CODE

print(f"Result: y_hat = {b0} x0 + {b1} x1 + {b2} x2 + {b3} x3\n + {b4} x4 + {b5} x5 + {b6} x6 \n")
print(f"Total process time: {round(time.time() - t2,3)}")

weights_gd_rounded = np.array([b0,b1,b2,b3,b4,b5,b6])
print(f"The estimated weights using the gradient descent solution:\n {weights_gd_rounded.round(3)}")

### 1.3 Calculating the weights using PyTorch Tensors and gradient descent (25pts)

Here you are asked to implement the same manual gradient algorithm as above.  
However, this time please use PyTorch tensors and operators (instead of numpy).  

In [ ]:
#### SOLUTION REQUIRED ####

# Creating data and target tensors:
tX = torch.tensor(X, dtype=torch.float32)
tY = torch.tensor(Y.to_numpy(), dtype=torch.float32)

t3 = time.time()

# Randomly initialize weights using PyTorch
tb0 = # FILL IN THE MISSING CODE
tb1 = # FILL IN THE MISSING CODE
tb2 = # FILL IN THE MISSING CODE
tb3 = # FILL IN THE MISSING CODE
tb4 = # FILL IN THE MISSING CODE
tb5 = # FILL IN THE MISSING CODE
tb6 = # FILL IN THE MISSING CODE

# Set learning rate
learning_rate = # FILL IN THE MISSING CODE

# Set number of epochs:
epochs = int(5e3)

# Loop through epochs, and apply gradient descent to update the weights:
for t in range(epochs):
    # Forward pass: compute predicted y
    # y_pred = b0x0 + b1x1 + b2x2 + b3x3 + b4x4 + b5x5 + b6x6
    tY_pred =# FILL IN THE MISSING CODE

    # Compute and print loss
    loss = # FILL IN THE MISSING CODE
    if t % 1000 == 0:
        print(f"epoch:{t}, loss:{loss}")

    # Backprop to compute gradients of betas with respect to loss
    grad_y_pred = # FILL IN THE MISSING CODE
    # print(grad_y_pred.sum())
    grad_b0 = # FILL IN THE MISSING CODE
    grad_b1 = # FILL IN THE MISSING CODE
    grad_b2 = # FILL IN THE MISSING CODE
    grad_b3 = # FILL IN THE MISSING CODE
    grad_b4 = # FILL IN THE MISSING CODE
    grad_b5 = # FILL IN THE MISSING CODE
    grad_b6 = # FILL IN THE MISSING CODE

    # Update weights
    tb0 -= # FILL IN THE MISSING CODE
    tb1 -= # FILL IN THE MISSING CODE
    tb2 -= # FILL IN THE MISSING CODE
    tb3 -= # FILL IN THE MISSING CODE
    tb4 -= # FILL IN THE MISSING CODE
    tb5 -= # FILL IN THE MISSING CODE
    tb6 -= # FILL IN THE MISSING CODE

print(f"Result: y_hat = {tb0} x0 + {tb1} x1 + {tb2} x2 + {tb3} x3\n + {tb4} x4 + {tb5} x5 + {tb6} x6 \n")
print(f"Total process time: {round(time.time() - t3,3)}")

weights_gd_tensors_rounded = np.array([tb0.item(),tb1.item(),tb2.item(),tb3.item(),
                                       tb4.item(),tb5.item(),tb6.item()])
print(f"The estimated weights using the gradient descent on tensors:\n {weights_gd_tensors_rounded.round(3)}")

### 1.4 Calculating the weights using PyTorch Autograd! (15pts)

Lastly, you can finally make a full transition to PyTorch autograd framework.   
Please replace your manual gradient calculations with PyTorch's automatic gradient tracking.   
In addition, instead of declaring the parameters separately, we will use one tensor to represent all of the model's parameters.  

In [ ]:
# FILL IN THE MISSING CODE

t4 = time.time()

# Randomly initialize weights using PyTorch.
# Use a single 1-D tensor of shape (7,) for all the weights. Make sure requires_grad=True.
tb = # FILL IN THE MISSING CODE


# Set learning rate
learning_rate = # FILL IN THE MISSING CODE
# Set number of epochs:
epochs = int(5e3)

# Loop through epochs, and apply gradient descent to update the weights:
for t in range(epochs):

    # Forward pass: compute predicted y
    tY_pred = # FILL IN THE MISSING CODE

    # Compute and print loss
    loss = # FILL IN THE MISSING CODE
    if t % 1000 == 0:
        print(f"epoch:{t}, loss:{loss.item()}")

    # Backprop to compute gradients of betas with respect to loss
    loss.backward()

    with torch.no_grad():
        # Update weights
        tb -= learning_rate * tb.grad
        tb.grad = None # manually clear gradient after update


print(f"Result: betas = {tb} \n")
print(f"Total process time: {round(time.time() - t4,3)}")

weights_autograd_rounded = tb.cpu().detach().numpy()
print(f"The estimated weights using the pytorch autograd:\n {weights_autograd_rounded.round(3)}")

### 1.5 Conclusion (15pts)

1. Which method is the most accurate in finding the "true" weights? Why? Is it always applicabale? Hint - you should think about it from a statistical theory point of view.
2. Why did we scale the data ?  (answer with respect to the model/algorithm)
3. Do you think our model specification produced good results?  Why or why not? Justify your answer.
4. What can you say about the total process times between methods?
5. Can you suggest some improvements to the model definition?
6. Can you suggest some improvements to the solution algorithm?

YOUR ANSWERS HERE

1. ANSWER   
2. ANSWER   
3. ANSWER   
4. ANSWER   
5. ANSWER
6. ANSWER

Download your .ipynb file and upload it to the session, then run the code chunk below and download the .html file in your submission on Moodle.

In [1]:
!jupyter nbconvert --to html PS1_Part3_Autograd_2026_ID_000000000.ipynb

[NbConvertApp] Converting notebook PS1_Part3_Autograd_2026_ID_000000000.ipynb to html
[NbConvertApp] Writing 334343 bytes to PS1_Part3_Autograd_2026_ID_000000000.html
